# Comment Toxicity Detection — Colab Version

**Dataset:** Jigsaw Toxic Comment Classification (Kaggle)  
**Model:** Bidirectional LSTM (Keras)  
**Labels (6, multi-label):** `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`

## How to use this notebook
1. Upload this `.ipynb` to Google Colab (File → Upload notebook)
2. **Runtime → Change runtime type → CPU** is fine (GPU optional, will be faster)
3. Run **Cell 1** to upload `train.csv` from your computer (you can also upload `test.csv` if you want)
4. Run every cell top-to-bottom — the full pipeline takes ~5–15 min
5. Run the last cell to download the trained model and artifacts as a zip

## 1. Upload the dataset

In [ ]:
# Upload train.csv (~69 MB). This step needs manual action.
from google.colab import files
uploaded = files.upload()   # pick train.csv
print('files received:', list(uploaded.keys()))

## 2. Imports and configuration

Every hyper-parameter is at the top so you can tweak it in one place.

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional, LSTM, Dense,
                                     Dropout, GlobalMaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('tensorflow version:', tf.__version__)

# ------ Configuration ------
TRAIN_PATH   = 'train.csv'
ARTIFACTS    = 'artifacts'
PLOTS        = os.path.join(ARTIFACTS, 'plots')

LABEL_COLS   = ['toxic', 'severe_toxic', 'obscene',
                'threat', 'insult', 'identity_hate']

SAMPLE_SIZE  = 50_000   # increase if runtime allows; 50k ~= 10 min on CPU
MAX_VOCAB    = 20_000
MAX_LEN      = 100
EMBED_DIM    = 64
BATCH_SIZE   = 256
EPOCHS       = 3
VAL_SPLIT    = 0.1
RANDOM_STATE = 42

os.makedirs(PLOTS, exist_ok=True)

## 3. Load and explore

In [ ]:
df = pd.read_csv(TRAIN_PATH)
print('shape :', df.shape)
print('nulls :', int(df.isna().sum().sum()))
print('duplicates :', int(df.duplicated(subset=['comment_text']).sum()))

label_pct = (df[LABEL_COLS].mean() * 100).round(2)
print('\nlabel distribution (positive class %):')
print(label_pct)

# Plot label distribution
fig, ax = plt.subplots(figsize=(8, 4))
label_pct.sort_values().plot(kind='barh', color='#ef4444', ax=ax)
ax.set_xlabel('% of comments')
ax.set_title('Label distribution (multi-label)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, '01_label_distribution.png'), dpi=120)
plt.show()

lens = df['comment_text'].str.split().str.len()
print(f'\ncomment length (words)  mean={lens.mean():.0f}  '
      f'median={lens.median():.0f}  95th_pct={lens.quantile(0.95):.0f}  '
      f'max={lens.max()}')

## 4. Clean text

Lowercase, strip URLs/HTML/digits/punctuation. Apostrophes are kept so `don't` / `can't` survive.

In [ ]:
_URL_RE        = re.compile(r'http\S+|www\.\S+')
_HTML_RE       = re.compile(r'<.*?>')
_NON_ALPHA_RE  = re.compile(r"[^a-z\s']")
_MULTI_SPACE   = re.compile(r'\s+')

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = _URL_RE.sub(' ', text)
    text = _HTML_RE.sub(' ', text)
    text = _NON_ALPHA_RE.sub(' ', text)
    text = _MULTI_SPACE.sub(' ', text).strip()
    return text

df['comment_text'] = df['comment_text'].astype(str).apply(clean_text)
df = df[df['comment_text'].str.len() > 0].reset_index(drop=True)
print('after cleaning:', df.shape)

# Subsample for CPU-friendly training
if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'using random sample of {SAMPLE_SIZE} rows')

## 5. Tokenize and vectorize

In [ ]:
X_text = df['comment_text'].tolist()
y = df[LABEL_COLS].values.astype('float32')

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_text, y, test_size=VAL_SPLIT, random_state=RANDOM_STATE)

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

def vectorize(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts),
                         maxlen=MAX_LEN, padding='post', truncating='post')

X_train = vectorize(X_train_text)
X_val   = vectorize(X_val_text)
print('X_train shape:', X_train.shape, ' X_val shape:', X_val.shape)

## 6. Build the Bidirectional LSTM model

**Why these layers:**
- **Embedding** — turns each token id into a dense learnable vector
- **Bidirectional LSTM** — reads the comment forward *and* backward to catch context (`not a bad guy` vs `a bad guy`)
- **GlobalMaxPooling1D** — picks the strongest signal across all time steps
- **Sigmoid output × 6** — multi-label, each label is an independent yes/no
- **binary_crossentropy loss** — the right loss for multi-label sigmoid outputs

In [ ]:
model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(LABEL_COLS), activation='sigmoid'),
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=True)],
)
model.summary()

## 7. Train

**On CPU**, 50k samples × 3 epochs ≈ 10–15 minutes.  
**On GPU** (Runtime → Change runtime type → T4 GPU), ≈ 1–2 minutes.  
EarlyStopping will stop early if validation loss stops improving.

In [ ]:
early = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early],
    verbose=1,
)

# Training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history['loss'],     label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history.history['auc'],      label='train')
ax[1].plot(history.history['val_auc'],  label='val')
ax[1].set_title('ROC-AUC'); ax[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, '02_training_curves.png'), dpi=120)
plt.show()

## 8. Evaluate on validation set

**Why per-label metrics?** Because the labels are imbalanced — `threat` is only ~0.3 % of comments, so global accuracy is misleading. Per-label precision, recall, F1 and ROC-AUC tell the real story.

In [ ]:
THRESHOLD = 0.5
probs = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
preds = (probs >= THRESHOLD).astype(int)

rows = []
for i, lbl in enumerate(LABEL_COLS):
    rows.append({
        'label': lbl,
        'accuracy':  round(accuracy_score(y_val[:, i], preds[:, i]), 4),
        'precision': round(precision_score(y_val[:, i], preds[:, i], zero_division=0), 4),
        'recall':    round(recall_score(y_val[:, i], preds[:, i], zero_division=0), 4),
        'f1':        round(f1_score(y_val[:, i], preds[:, i], zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y_val[:, i], probs[:, i]), 4),
    })
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))
print(f"\nmacro ROC-AUC = {metrics['roc_auc'].mean():.4f}")

metrics.to_csv(os.path.join(ARTIFACTS, 'metrics.csv'), index=False)

## 9. Save the trained artifacts

In [ ]:
model.save(os.path.join(ARTIFACTS, 'toxicity_model.keras'))
with open(os.path.join(ARTIFACTS, 'tokenizer.json'), 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
with open(os.path.join(ARTIFACTS, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump({'max_len': MAX_LEN, 'labels': LABEL_COLS}, f, indent=2)
print('saved artifacts to', ARTIFACTS)
!ls -la {ARTIFACTS}

## 10. Quick sanity check — try some sample comments

In [ ]:
def predict_text(text, threshold=0.5):
    cleaned = clean_text(text)
    seq = pad_sequences(tokenizer.texts_to_sequences([cleaned]),
                        maxlen=MAX_LEN, padding='post', truncating='post')
    p = model.predict(seq, verbose=0)[0]
    return {lbl: round(float(p[i]), 4) for i, lbl in enumerate(LABEL_COLS)}

samples = [
    'Thanks for the helpful explanation, I really appreciate it.',
    'You are a complete idiot and I hate you so much.',
    'I think we should discuss this in more detail tomorrow.',
    'Go kill yourself you worthless piece of trash.',
]
for s in samples:
    print('>', s)
    print(' ', predict_text(s))
    print()

## 11. (Optional) Gradio mini-app inside the notebook

Since Streamlit is painful in Colab, use Gradio — one line gives you a public URL you can paste into LinkedIn demo videos.

In [ ]:
!pip install -q gradio
import gradio as gr

def classify(comment):
    probs = predict_text(comment)
    is_toxic = any(p >= 0.5 for p in probs.values())
    verdict = '⚠️ TOXIC' if is_toxic else '✅ Clean'
    return verdict, probs

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=3, placeholder='Paste a comment ...'),
    outputs=[gr.Label(label='Verdict'), gr.Label(label='Probabilities')],
    title='Comment Toxicity Detector',
    description='Bidirectional LSTM — predicts 6 toxicity labels.',
    examples=[
        ['Thanks, this is really helpful!'],
        ['You are a complete idiot.'],
        ['I disagree with your point but respect your view.'],
    ],
)
demo.launch(share=True)

## 12. Download artifacts (zip)

Save everything the Streamlit app needs (model + tokenizer + config + metrics) as a zip you can download to your laptop.

In [ ]:
import shutil
shutil.make_archive('toxicity_artifacts', 'zip', ARTIFACTS)
from google.colab import files
files.download('toxicity_artifacts.zip')